# 02 — Extração e rotulagem de patches (reconstruído)

**O que este notebook faz**: lê o dataset bruto (imagens + labels YOLO), varre cada imagem com uma janela deslizante de 48×48 px, e rotula cada patch como `pos` (cacho) ou `neg` (não-cacho) com base na fração do patch coberta por alguma *bounding box* anotada. Salva tudo em `patches_{split}_48.npz`, formato consumido pelo notebook 03.

## Configuração final (conforme decisões da fase de calibração)

| Parâmetro | Valor | Justificativa |
|---|---|---|
| Tamanho do patch | 48×48 | 32×32 foi abandonado (rotulagem ruidosa) |
| Stride | 24 | 50% de sobreposição |
| `pos_thr` | **0.35** | patch é `pos` se ≥ 35% da sua área cai dentro de alguma bbox |
| `neg_thr` | **0.05** | patch é `neg` se < 5% da área cai dentro de bbox |
| zona ambígua | 0.05 – 0.35 | descartada (reduz ruído de rótulo) |
| imagens sem label | usadas | fonte de negativos "limpos" |
| undersampling | só no `train`, 2 neg : 1 pos | evita saturação do discriminador negativo |
| `valid` / `test` | distribuição natural | refletem a prevalência real (~3% pos) |

**Reprodutibilidade**: o undersampling usa `np.random.default_rng(SEED)` com `SEED=42`. Como este notebook foi reconstruído da especificação (não é o arquivo bit-a-bit original), os números podem sair *ligeiramente* diferentes dos históricos (ex.: ~23.859 positivos no train) por amostragem distinta — isso é esperado e não indica erro. A ordem de varredura das imagens é ordenada alfabeticamente para determinismo.

## 0. Setup — ajuste o caminho do dataset aqui

In [1]:
from pathlib import Path
import os
import numpy as np
import cv2
import pandas as pd
from tqdm import tqdm

# ====================================================================
#  AJUSTE ESTE CAMINHO PARA A PASTA QUE CONTÉM  test/  train/  valid/
# ====================================================================
DATASET_ROOT = Path(r"C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\dataset")

PATCHES_DIR = Path("./patches")
PATCHES_DIR.mkdir(exist_ok=True)

# --- Hiperparâmetros (configuração final) ---
PATCH_SIZE = 32
STRIDE     = 16
POS_THR    = 0.5
NEG_THR    = 0.1
NEG_PER_POS = 2          # undersampling no train: 2 neg : 1 pos
SEED       = 42
SPLITS     = ["train", "valid", "test"]

# Subestrutura esperada dentro de cada split (padrão Roboflow YOLO)
IMAGES_SUBDIR = "images"
LABELS_SUBDIR = "labels"
IMG_EXTS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

print("Dataset root:", DATASET_ROOT)
print("Existe?", DATASET_ROOT.exists())
if DATASET_ROOT.exists():
    for s in SPLITS:
        imgs = DATASET_ROOT / s / IMAGES_SUBDIR
        lbls = DATASET_ROOT / s / LABELS_SUBDIR
        print(f"  {s}: images={'OK' if imgs.exists() else 'FALTA'} "
              f"({imgs})  labels={'OK' if lbls.exists() else 'FALTA'}")
print(f"\nPatch {PATCH_SIZE}x{PATCH_SIZE}, stride {STRIDE}, "
      f"pos_thr={POS_THR}, neg_thr={NEG_THR}")

Dataset root: C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\dataset
Existe? True
  train: images=OK (C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\dataset\train\images)  labels=OK
  valid: images=OK (C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\dataset\valid\images)  labels=OK
  test: images=OK (C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\dataset\test\images)  labels=OK

Patch 32x32, stride 16, pos_thr=0.5, neg_thr=0.1


### Checagem de sanidade do caminho

Se a célula acima mostrou `FALTA` para `images`/`labels`, verifique a subestrutura. Alguns exports do Roboflow usam `train/images` + `train/labels` (esperado aqui); outros colocam as imagens direto em `train/`. Rode a célula abaixo para inspecionar o que há dentro de um split.

In [2]:
def peek(split="train", n=5):
    base = DATASET_ROOT / split
    if not base.exists():
        print(f"{base} não existe.")
        return
    print(f"Conteúdo de {base}:")
    for item in sorted(base.iterdir())[:10]:
        tag = "[dir]" if item.is_dir() else "     "
        print(f"  {tag} {item.name}")
    imgs_dir = base / IMAGES_SUBDIR
    if imgs_dir.exists():
        imgs = [p for p in sorted(imgs_dir.iterdir()) if p.suffix in IMG_EXTS]
        print(f"\n  {len(imgs)} imagens em {imgs_dir}")
        for p in imgs[:n]:
            print(f"    {p.name}")

peek("train")

Conteúdo de C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\dataset\train:
  [dir] images
  [dir] labels

  936 imagens em C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\dataset\train\images
    GX010691_MP4-0001_jpg.rf.b06634b6280f9635c6a58cb499db3269.jpg
    GX010691_MP4-0001_jpg.rf.e83118d67cdd55e8c6b686dcde14d208.jpg
    GX010691_MP4-0001_jpg.rf.f364f6bddd1a3c610bbca2ebb9428180.jpg
    GX010691_MP4-0002_jpg.rf.c53504d7458e4d5ae25c9e3adee5c10a.jpg
    GX010691_MP4-0002_jpg.rf.d9fc68967fa63a08211df255043e4be4.jpg


## 1. Funções utilitárias

- `load_yolo_labels`: lê um `.txt` YOLO e devolve bboxes em pixels `(x1,y1,x2,y2)`.
- `patch_coverage`: maior fração da área do patch coberta por **alguma** bbox.
- `iter_image_label_pairs`: pareia cada imagem ao seu label (label ausente = imagem sem cachos = fonte de negativos).

In [3]:
def load_yolo_labels(label_path, img_w, img_h):
    """Lê arquivo .txt YOLO -> lista de bboxes em pixels (x1,y1,x2,y2).
    Arquivo ausente ou vazio -> lista vazia (imagem sem cachos)."""
    bboxes = []
    if not os.path.exists(label_path):
        return bboxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            # formato: classe xc yc w h  (normalizados 0..1)
            _, xc, yc, w, h = map(float, parts[:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            bboxes.append((x1, y1, x2, y2))
    return bboxes


def patch_coverage(patch_box, bboxes):
    """Maior fração da área do patch coberta por alguma bbox."""
    px1, py1, px2, py2 = patch_box
    patch_area = (px2 - px1) * (py2 - py1)
    best = 0.0
    for (x1, y1, x2, y2) in bboxes:
        ix1, iy1 = max(px1, x1), max(py1, y1)
        ix2, iy2 = min(px2, x2), min(py2, y2)
        if ix2 > ix1 and iy2 > iy1:
            inter = (ix2 - ix1) * (iy2 - iy1)
            best = max(best, inter / patch_area)
    return best


def iter_image_label_pairs(split):
    """Gera (img_path, label_path) para um split, ordenado por nome.
    label_path pode não existir (imagem sem cachos)."""
    images_dir = DATASET_ROOT / split / IMAGES_SUBDIR
    labels_dir = DATASET_ROOT / split / LABELS_SUBDIR
    img_paths = sorted(
        [p for p in images_dir.iterdir() if p.suffix in IMG_EXTS],
        key=lambda p: p.name,
    )
    for img_path in img_paths:
        label_path = labels_dir / (img_path.stem + ".txt")
        yield img_path, label_path

## 2. Extração por split

Para cada imagem: varre a grade de patches, calcula a cobertura, aplica a regra de rótulo. Coleta separadamente positivos e negativos; o undersampling é aplicado depois, **só no train**.

In [4]:
def extract_split(split, verbose=True):
    """Retorna (X_pos, X_neg) como arrays uint8 (N, 48, 48, 3) em BGR,
    além de contadores de diagnóstico."""
    pos_patches, neg_patches = [], []
    n_images = 0
    n_empty_label = 0
    n_ambiguous = 0

    pairs = list(iter_image_label_pairs(split))
    for img_path, label_path in tqdm(pairs, desc=f"{split}", disable=not verbose):
        img = cv2.imread(str(img_path))  # BGR
        if img is None:
            continue
        n_images += 1
        h, w = img.shape[:2]
        bboxes = load_yolo_labels(str(label_path), w, h)
        if len(bboxes) == 0:
            n_empty_label += 1

        for y in range(0, h - PATCH_SIZE + 1, STRIDE):
            for x in range(0, w - PATCH_SIZE + 1, STRIDE):
                patch_box = (x, y, x + PATCH_SIZE, y + PATCH_SIZE)
                cov = patch_coverage(patch_box, bboxes)
                if cov >= POS_THR:
                    label = "pos"
                elif cov < NEG_THR:
                    label = "neg"
                else:
                    n_ambiguous += 1
                    continue  # zona ambígua: descarta
                patch = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                if patch.shape[:2] != (PATCH_SIZE, PATCH_SIZE):
                    continue
                if label == "pos":
                    pos_patches.append(patch)
                else:
                    neg_patches.append(patch)

    X_pos = (np.stack(pos_patches).astype(np.uint8)
             if pos_patches else np.empty((0, PATCH_SIZE, PATCH_SIZE, 3), np.uint8))
    X_neg = (np.stack(neg_patches).astype(np.uint8)
             if neg_patches else np.empty((0, PATCH_SIZE, PATCH_SIZE, 3), np.uint8))

    diag = {
        "split": split,
        "n_images": n_images,
        "n_empty_label": n_empty_label,
        "n_pos": len(X_pos),
        "n_neg": len(X_neg),
        "n_ambiguous": n_ambiguous,
        "razao_neg_pos": round(len(X_neg) / max(1, len(X_pos)), 2),
    }
    return X_pos, X_neg, diag

In [5]:
raw = {}      # split -> (X_pos, X_neg)
diagnostics = []

for split in SPLITS:
    X_pos, X_neg, diag = extract_split(split)
    raw[split] = (X_pos, X_neg)
    diagnostics.append(diag)
    print(f"  {split}: pos={diag['n_pos']}  neg={diag['n_neg']}  "
          f"razao={diag['razao_neg_pos']}  ambiguos={diag['n_ambiguous']}  "
          f"(imgs={diag['n_images']}, vazias={diag['n_empty_label']})")

diag_df = pd.DataFrame(diagnostics)
diag_df

train: 100%|██████████| 936/936 [00:47<00:00, 19.81it/s]


  train: pos=46485  neg=1278610  razao=27.51  ambiguos=98561  (imgs=936, vazias=36)


valid: 100%|██████████| 88/88 [00:05<00:00, 14.85it/s]


  valid: pos=3169  neg=122033  razao=38.51  ambiguos=8646  (imgs=88, vazias=0)


test: 100%|██████████| 45/45 [00:01<00:00, 23.78it/s]


  test: pos=1482  neg=62249  razao=42.0  ambiguos=4714  (imgs=45, vazias=2)


,split,n_images,n_empty_label,n_pos,n_neg,n_ambiguous,razao_neg_pos
0,train,936,36,46485,1278610,98561,27.51
1,valid,88,0,3169,122033,8646,38.51
2,test,45,2,1482,62249,4714,42.00


## 3. Undersampling (só no `train`) e montagem dos arrays finais

- `train`: mantém todos os positivos e sorteia `NEG_PER_POS × n_pos` negativos.
- `valid` / `test`: mantém a distribuição natural (todos os patches), porque é onde medimos performance e queremos refletir a prevalência real (~3% pos).

In [6]:
rng = np.random.default_rng(SEED)
final = {}

for split in SPLITS:
    X_pos, X_neg = raw[split]
    n_pos, n_neg = len(X_pos), len(X_neg)

    if split == "train":
        target_neg = min(n_neg, NEG_PER_POS * n_pos)
        sel = rng.choice(n_neg, size=target_neg, replace=False)
        X_neg_use = X_neg[np.sort(sel)]
        print(f"train: undersampling neg {n_neg} -> {target_neg} "
              f"(alvo {NEG_PER_POS}x{n_pos}={NEG_PER_POS*n_pos})")
    else:
        X_neg_use = X_neg
        print(f"{split}: distribuição natural mantida (pos={n_pos}, neg={n_neg})")

    X = np.concatenate([X_pos, X_neg_use], axis=0)
    y = np.array(["pos"] * len(X_pos) + ["neg"] * len(X_neg_use))

    # embaralha (importante para o train; inofensivo para valid/test)
    perm = rng.permutation(len(X))
    X, y = X[perm], y[perm]

    final[split] = (X, y)
    print(f"  -> {split}: X={X.shape}  pos={(y=='pos').sum()}  neg={(y=='neg').sum()}")

train: undersampling neg 1278610 -> 92970 (alvo 2x46485=92970)
  -> train: X=(139455, 32, 32, 3)  pos=46485  neg=92970
valid: distribuição natural mantida (pos=3169, neg=122033)
  -> valid: X=(125202, 32, 32, 3)  pos=3169  neg=122033
test: distribuição natural mantida (pos=1482, neg=62249)
  -> test: X=(63731, 32, 32, 3)  pos=1482  neg=62249


## 4. Salvar `patches_{split}_48.npz`

Formato exatamente como o notebook 03 espera: chaves `X` (uint8, N×48×48×3, BGR) e `y` (`'pos'`/`'neg'`).

In [7]:
for split in SPLITS:
    X, y = final[split]
    out_path = PATCHES_DIR / f"patches_{split}_{PATCH_SIZE}.npz"
    np.savez_compressed(out_path, X=X, y=y)
    size_mb = out_path.stat().st_size / 1e6
    print(f"salvo: {out_path}  ({X.shape}, {size_mb:.1f} MB)")

salvo: patches\patches_train_32.npz  ((139455, 32, 32, 3), 368.6 MB)
salvo: patches\patches_valid_32.npz  ((125202, 32, 32, 3), 345.8 MB)
salvo: patches\patches_test_32.npz  ((63731, 32, 32, 3), 176.0 MB)


## 5. Sanidade — recarregar e conferir

Confere que os arquivos abrem, têm as chaves certas e o shape esperado. É isto que o notebook 03 vai consumir.

In [8]:
for split in SPLITS:
    data = np.load(PATCHES_DIR / f"patches_{split}_{PATCH_SIZE}.npz")
    X, y = data["X"], data["y"]
    n_pos = int((y == "pos").sum())
    n_neg = int((y == "neg").sum())
    assert X.dtype == np.uint8, f"{split}: X deveria ser uint8"
    assert X.shape[1:] == (PATCH_SIZE, PATCH_SIZE, 3), f"{split}: shape inesperado {X.shape}"
    assert len(X) == len(y), f"{split}: X e y com tamanhos diferentes"
    print(f"{split:<6}  X={str(X.shape):<22}  pos={n_pos:<7}  neg={n_neg:<7}  OK")

print("\nTudo certo. Próximo: notebook 03 (binarização thermometer-HSV).")

train   X=(139455, 32, 32, 3)     pos=46485    neg=92970    OK
valid   X=(125202, 32, 32, 3)     pos=3169     neg=122033   OK
test    X=(63731, 32, 32, 3)      pos=1482     neg=62249    OK

Tudo certo. Próximo: notebook 03 (binarização thermometer-HSV).


## 6. Notas para o relatório

- A regra de rótulo por **cobertura do patch** (não IoU bbox-a-bbox) foi escolhida porque a tarefa é classificação de patch, não casamento de caixas: o que importa é *quanto do patch é cacho*.
- A **zona ambígua descartada** (0,05–0,35) é uma decisão de projeto que troca quantidade por qualidade de rótulo — patches meio-cacho confundem o treino do classificador.
- O **undersampling 2:1 só no train** equilibra o aprendizado sem distorcer a avaliação: `valid`/`test` mantêm a prevalência real (~3% pos), o que torna o F1 a métrica honesta (acurácia seria enganosa nesse desbalanceamento).
- Como este notebook foi **reconstruído da especificação**, pequenas diferenças nos totais (vs. ~23.859 pos históricos no train) são esperadas por reamostragem; a metodologia é idêntica.